
# Tutorial 3 — `measurement.py` (live g² processing)

**Prerequisites:** `hardware_tutorial.ipynb`, `acquisition_tutorial.ipynb`

---

## What is `measurement.py`?

When a photon arrives, the TimeTagger timestamps it. For g² we need to know:

- How many photons on each detector? (**singles**)
- How many **coincidences** between pairs?
- What does the **correlation histogram** look like?

Two strategies:

| Strategy | Where math runs | Output size |
|----------|-----------------|-------------|
| Save raw tags → analyse later | Your laptop | Huge `.ttbin`, flexible |
| **Live reduction (this module)** | TimeTagger FPGA + driver | Small `.pkl` |

`CorrelationRecorder` plugs into `Acquisition` exactly like `RawTimeTagRecorder`.

---

## Our 6-channel LOA setup (reminder)

| Channel | Label | Meaning |
|---------|-------|---------|
| 1 | H3T | Harmonic 3, transmitted arm |
| 2 | H3R | Harmonic 3, reflected arm |
| 3 | H4T | Harmonic 4, transmitted |
| 4 | H4R | Harmonic 4, reflected |
| … | … | … |

**Virtual channel H3** = H3T + H3R merged (either detector fired).



# Bootstrap — run this cell first

This finds the project folder and lets Python import our code from `src/`.

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "hardware.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:

# Bootstrap — run this cell first

This finds the project folder and lets Python import our code from `src/`.

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "hardware.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:

## Lab vs laptop

| Flag | When to use |
|------|-------------|
| `LIVE_HARDWARE = False` | At home, on GitHub, learning the API (default) |
| `LIVE_HARDWARE = True` | On the **lab PC**, instruments plugged in |

Set it in the next cell before running hardware cells.

LIVE_HARDWARE = False   # <-- change to True on the lab PC

# --- edit these for YOUR bench ---
TT_SERIAL = ""              # TimeTagger serial, or "" for first device
PRM1_SERIAL = "27264707"    # Thorlabs K-Cube serial (string)
ELL14_ADDRESS = 2           # Elliptec address (integer 0-9)
CHANNELS = [1, 2, 3, 4, 5, 6]



---
# Part A — Two measurement modes

| `experiment_type` | What is computed |
|-------------------|------------------|
| `"g2"` | Physical singles, 15 pair coincidences, 15 correlation histograms |
| `"g2_heralded_virtual"` | All of g2 **plus** virtual harmonic channels, heralded 3-fold correlations |

Set via `AcquisitionConfig(experiment_type="g2_heralded_virtual")` or `CorrelationRecorder(mode="g2")`.



---
# Part B — `CorrelationRecorder`

```python
CorrelationRecorder(
    mode=None,                      # None = read from config experiment_type
    record_physical_histograms=True # strongly recommended for hbt_core
)
```

## Main method: `record(device, duration_ps, savepath, params=..., logger=...)`

Called by `Acquisition` — you rarely call it yourself.

**Steps inside `record()`:**

1. Read channels, binwidth, coincidence window from `params`
2. If virtual mode: build `Combiner` / `Coincidence` virtual channels
3. Create `Countrate`, `Coincidences`, `Correlation` measurements
4. `startFor(duration)` → wait → read results
5. Save `{"Parameters": params, "data": results}` to `<savepath>.pkl`

## Helper: `build_modes(channels, mode_on_channel)`

Groups channels by harmonic:

```python
CorrelationRecorder.build_modes(
    [1,2,3,4,5,6],
    ["H3T","H3R","H4T","H4R","H5T","H5R"]
)
# -> {"H3": [1,2], "H4": [3,4], "H5": [5,6]}
```


In [ ]:

from src.measurement import CorrelationRecorder

modes = CorrelationRecorder.build_modes(
    [1, 2, 3, 4, 5, 6],
    ["H3T", "H3R", "H4T", "H4R", "H5T", "H5R"],
)
print("Harmonic groups:", modes)

rec = CorrelationRecorder(mode="g2_heralded_virtual", record_physical_histograms=True)
print("Recorder name:", rec.name)



---
# Part C — What is inside the `.pkl` file?

The `"data"` block (read by `hbt_core.HBTMeasurement`):

### Always (physical)

| Key | Content |
|-----|---------|
| `counts_physical` | Total singles per detector `{"1": 12345, ...}` |
| `countrates_physical` | counts/s per detector |
| `coincidences_twofold_physical` | Integrated coincidences per pair `"(1,2)": 999` |
| `correlations_physical` | Full histograms per pair (if enabled) |

### Extra in `g2_heralded_virtual`

| Key | Content |
|-----|---------|
| `counts_virtual` / `countrates_virtual` | Per harmonic H3, H4, H5 |
| `coincidences_twofold_virtual` | e.g. `"(H3,H4)": 123` |
| `correlations_virtual` | Histograms between harmonics |
| `heralded_threefold` | Herald permutations with numerator/denominator histograms |

Each correlation entry looks like:
```python
{"time_bins": [...], "counts": [...]}
```


In [ ]:

# Offline: show structure without hardware
example_keys = [
    "counts_physical", "countrates_physical",
    "coincidences_twofold_physical", "correlations_physical",
    "counts_virtual", "correlations_virtual", "heralded_threefold",
]
for k in example_keys:
    print(" ", k)
print("\nLoad real data later with:")
print("  from src.hbt_core import HBTMeasurement")
print("  m = HBTMeasurement('path/to/file.pkl')")



---
# Part D — `merge()` — combining chunks

When `Acquisition.run_chunked()` runs, each chunk produces a result dict. `merge()` adds them:

| Quantity | Rule |
|----------|------|
| Counts, coincidences | **Sum** |
| Count rates | **Duration-weighted average** |
| Histogram bins | **Sum bin-by-bin** |
| Total duration | **Sum** |

First chunk: `merge(None, chunk)` → copy of chunk.

This is how `MERGED.pkl` accumulates statistics without keeping every chunk in RAM forever.


In [ ]:

from src.measurement import CorrelationRecorder

r = CorrelationRecorder()
a = {"duration_ps": 1000, "counts_physical": {"1": 100}, "countrates_physical": {"1": 10.0}}
b = {"duration_ps": 1000, "counts_physical": {"1": 200}, "countrates_physical": {"1": 20.0}}
m = r.merge(r.merge(None, a), b)
print("Merged counts ch1:", m["counts_physical"]["1"])      # 300
print("Merged rate ch1:", m["countrates_physical"]["1"])    # 15.0 (average)
print("Merged duration_ps:", m["duration_ps"])                # 2000



---
# Part E — `coincidence_threshold_stop()` — when to stop measuring

Factory function → returns a **stop condition** for `Acquisition`.

```python
stop = coincidence_threshold_stop(min_counts=100_000, kind="physical")
# kind can be "physical" or "virtual"
```

After each chunk, `Acquisition` calls:
```python
should_stop, reason = stop(merged_data)
```

- Returns `(True, "...")` when **every** pair in `coincidences_twofold_physical` has ≥ `min_counts`
- Returns `(False, "...")` while any pair is still below threshold

This replaces the old `experiment_v4.py` "measure until 100k coincidences" loop.


In [ ]:

from src.measurement import coincidence_threshold_stop

stop = coincidence_threshold_stop(min_counts=100_000, kind="physical")

fake_low = {"coincidences_twofold_physical": {"(1,2)": 5000, "(1,3)": 8000}}
fake_high = {"coincidences_twofold_physical": {"(1,2)": 150_000, "(1,3)": 200_000}}

print("Low stats:", stop(fake_low))
print("High stats:", stop(fake_high))



---
# Part F — Full lab example (CorrelationRecorder + chunking)

Copy this to start a real g² heralded acquisition.


In [ ]:

from src.acquisition import (
    Acquisition, AcquisitionConfig, LaserParams, TimeTaggerParams,
    ChunkingParams, ScanPoint,
)
from src.measurement import CorrelationRecorder, coincidence_threshold_stop
from src.hardware import RotationStageController

cfg = AcquisitionConfig(
    base_dir=ROOT / "data",
    material="CdTe110",
    experiment_type="g2_heralded_virtual",
    laser=LaserParams(rep_rate_hz=18.66e6, wavelength_nm=2100),
    timetagging=TimeTaggerParams(
        channels=[1, 2, 3, 4, 5, 6],
        mode_on_channel=["H3T", "H3R", "H4T", "H4R", "H5T", "H5R"],
        trigger_levels_v=0.5,
        delays_ps=[0, -1000, 20000, 16500, 17800, 17300],
        binwidth_ps=100,
        num_bins=5000,
        coincidence_window_ps=25000,
    ),
    chunking=ChunkingParams(
        enabled=True,
        chunk_minutes=2,
        max_chunks=100,
        stop_when_reached=True,
    ),
)

if LIVE_HARDWARE:
    stages = RotationStageController([PRM1_SERIAL]).connect()
    stop = coincidence_threshold_stop(min_counts=100_000, kind="physical")
    with Acquisition(
        cfg,
        recorder=CorrelationRecorder(record_physical_histograms=True),
        stages=stages,
        stop_condition=stop,
    ) as acq:
        acq.run_scan([ScanPoint(power_mw=42.0, angle_deg=55.0, stage_id=PRM1_SERIAL)])
    print("Done. Analyse MERGED/*.pkl with hbt_core.HBTMeasurement")
else:
    print("[offline] Full pipeline:")
    print("  1. setup: connect tagger, set triggers/delays")
    print("  2. rotate HWP to ScanPoint.angle_deg")
    print("  3. loop chunks: CorrelationRecorder.record -> merge")
    print("  4. stop when coincidences >= threshold")
    print("  5. analyse MERGED.pkl with hbt_core")



---
# Part G — How this connects to analysis (`hbt_core.py`)

```python
from src.hbt_core import HBTMeasurement

m = HBTMeasurement("data/.../MERGED/P42_num0_MERGED.pkl")
g2 = m.compute_g2_direct(1, 3)           # scalar physical g²
R  = m.compute_R_parameter(g2_cross, g2_a, g2_b)
```

The pickle schema from `CorrelationRecorder` matches what `HBTMeasurement` expects — no conversion step.

---

# Summary

| Component | Role |
|-----------|------|
| `CorrelationRecorder` | Live g² / coincidence / herald processing |
| `build_modes()` | Group T/R detectors into harmonics |
| `merge()` | Add chunk results together |
| `coincidence_threshold_stop()` | Smart early stopping |
| `Acquisition` + `CorrelationRecorder` | Full automated experiment |

**Pipeline complete:** hardware → acquisition → measurement → hbt_core analysis.
